# V18-C9: 5 Critical Methodological Fixes
## GSE182159 Tissue-Separated Reanalysis — ITLAS Analytical Workflow

| Fix | Issue | Severity | Output |
|-----|-------|----------|--------|
| **Fix 1** | FDR (Benjamini-Hochberg) correction | 🔴 CRITICAL | C3/C5/C4/C7 q-values |
| **Fix 2** | Add 3 missing pathways | 🟡 HIGH | antigen_presentation, type1_ifn, tgfb_signaling |
| **Fix 3** | "99999%" fold-change donor review | 🟡 HIGH | Reliability flags per gene |
| **Fix 4** | C3 (196) vs C5 (148) gene reconciliation | 🟡 HIGH | Source attribution |
| **Fix 5** | C7 correlation bootstrap CI | 🟡 HIGH | 95% CI + scatter plots |

**Input:** Existing C3–C8 CSV results + `GSE182159_gut2021_annotated.h5ad`
**Output:** `version18-analysis/C9_method_fixes/`

> ⚠️ **RULES:** Liver/Blood separate | dot/box only | Liver=red● Blood=blue▲ | No combined graphs

In [1]:
# ============================================================
# CELL 0: GPU SETUP & VERIFICATION
# ============================================================
!pip install scanpy anndata matplotlib seaborn scipy -q

import subprocess
import sys

print("=" * 70)
print("  V18 C3: GPU ENVIRONMENT SETUP")
print("=" * 70)

# GPU detection
try:
    result = subprocess.run(['nvidia-smi', '--query-gpu=name,memory.total,compute_cap',
                            '--format=csv,noheader'], capture_output=True, text=True)
    gpu_info = result.stdout.strip()
    print(f"✅ GPU detected: {gpu_info}")
except:
    print("⚠️ No GPU detected — will use CPU fallback")

# Install CuPy for GPU acceleration
try:
    import cupy as cp
    print(f"✅ CuPy {cp.__version__} ready")
    print(f"   GPU memory: {cp.cuda.Device(0).mem_info[1] / 1e9:.1f} GB total")
except ImportError:
    print("📦 Installing CuPy...")
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', 'cupy-cuda12x', '-q'])
    import cupy as cp
    print(f"✅ CuPy {cp.__version__} installed")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.1/2.1 MB 52.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 176.6/176.6 kB 18.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.1/60.1 kB 5.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 284.1/284.1 kB 28.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.2/9.2 MB 117.2 MB/s eta 0:00:00
  V18 C3: GPU ENVIRONMENT SETUP
✅ GPU detected: NVIDIA A100-SXM4-80GB, 81920 MiB, 8.0
✅ CuPy 14.0.1 ready
   GPU memory: 85.1 GB total


## Cell 1. Setup & Google Drive Mount

In [2]:
# ============================================================
#  CELL 1: SETUP & MOUNT
# ============================================================
print("=" * 70)
print("  C9: METHODOLOGICAL FIXES — SETUP")
print("=" * 70)

from google.colab import drive
drive.mount('/content/drive')

import os
import numpy as np
import pandas as pd
from scipy import stats
from statsmodels.stats.multitest import multipletests
import warnings
warnings.filterwarnings('ignore')

BASE = '/content/drive/MyDrive/ITLAS/results/version18-analysis'
DATA_PATH = '/content/drive/MyDrive/ITLAS/data/processed/GSE182159_gut2021_annotated.h5ad'

C3_DIR = f'{BASE}/C3_gene_expression/'
C4_DIR = f'{BASE}/C4_pathway/'
C5_DIR = f'{BASE}/C5_genes/'
C6_DIR = f'{BASE}/C6_discrepancy/'
C7_DIR = f'{BASE}/C7_patterns/'
C8_DIR = f'{BASE}/C8_characteristics/'

C9_DIR = f'{BASE}/C9_method_fixes/'
os.makedirs(C9_DIR, exist_ok=True)
os.makedirs(f'{C9_DIR}/Fix1_FDR/', exist_ok=True)
os.makedirs(f'{C9_DIR}/Fix2_NewPathways/', exist_ok=True)
os.makedirs(f'{C9_DIR}/Fix3_ExtremeFC/', exist_ok=True)
os.makedirs(f'{C9_DIR}/Fix4_GeneReconcile/', exist_ok=True)
os.makedirs(f'{C9_DIR}/Fix5_BootstrapCI/', exist_ok=True)

for d, name in [(C3_DIR,'C3'), (C4_DIR,'C4'), (C5_DIR,'C5'),
                (C6_DIR,'C6'), (C7_DIR,'C7'), (C8_DIR,'C8')]:
    files = os.listdir(d) if os.path.exists(d) else []
    print(f"  {name}: {len(files)} files → {d}")
    for f in sorted(files)[:5]:
        print(f"      {f}")

print(f"\n  C9 output → {C9_DIR}")
print("✅ Setup complete")

  C9: METHODOLOGICAL FIXES — SETUP
Mounted at /content/drive
  C3: 9 files → /content/drive/MyDrive/ITLAS/results/version18-analysis/C3_gene_expression/
      C3_NL_vs_IT_all.csv
      C3_NL_vs_IT_significant.csv
      C3_all_statistics.csv.gz
      C3_blood_donor_gene_expression.csv.gz
      C3_gene_list_196genes.csv
  C4: 4 files → /content/drive/MyDrive/ITLAS/results/version18-analysis/C4_pathway/
      C4_pathway_blood.csv
      C4_pathway_liver.csv
      C4_selected_genes_for_C5.csv
      C4_selected_pathways_for_C5.csv
  C5: 4 files → /content/drive/MyDrive/ITLAS/results/version18-analysis/C5_genes/
      C5_all_significant_genes.csv
      C5_gene_significance_ranking.csv
      C5_genes_blood.csv
      C5_genes_liver.csv
  C6: 2 files → /content/drive/MyDrive/ITLAS/results/version18-analysis/C6_discrepancy/
      C6_gene_discrepancy_master.csv
      C6_pathway_discrepancy_master.csv
  C7: 2 files → /content/drive/MyDrive/ITLAS/results/version18-analysis/C7_patterns/
      C7_gene

## Cell 2. Load Existing C3–C8 Results

In [3]:
# ============================================================
#  CELL 2: LOAD EXISTING RESULTS
# ============================================================
print("=" * 70)
print("  LOADING EXISTING C3-C8 RESULTS")
print("=" * 70)

# --- C3: All statistics (196 genes) ---
c3_stats_path = f'{C3_DIR}/C3_all_statistics.csv.gz'
if os.path.exists(c3_stats_path):
    c3_stats = pd.read_csv(c3_stats_path)
    print(f"  C3 stats: {c3_stats.shape} — columns: {list(c3_stats.columns[:10])}")
else:
    c3_stats_path2 = f'{C3_DIR}/C3_all_statistics.csv'
    if os.path.exists(c3_stats_path2):
        c3_stats = pd.read_csv(c3_stats_path2)
        print(f"  C3 stats: {c3_stats.shape}")
    else:
        print("  ⚠️ C3_all_statistics not found")
        c3_stats = None

# --- C4: Pathway results ---
c4_liver = pd.read_csv(f'{C4_DIR}/C4_pathway_liver.csv') if os.path.exists(f'{C4_DIR}/C4_pathway_liver.csv') else None
c4_blood = pd.read_csv(f'{C4_DIR}/C4_pathway_blood.csv') if os.path.exists(f'{C4_DIR}/C4_pathway_blood.csv') else None
if c4_liver is not None: print(f"  C4 Liver: {c4_liver.shape}")
if c4_blood is not None: print(f"  C4 Blood: {c4_blood.shape}")

# --- C5: Gene results (148 genes) ---
c5_liver = pd.read_csv(f'{C5_DIR}/C5_genes_liver.csv') if os.path.exists(f'{C5_DIR}/C5_genes_liver.csv') else None
c5_blood = pd.read_csv(f'{C5_DIR}/C5_genes_blood.csv') if os.path.exists(f'{C5_DIR}/C5_genes_blood.csv') else None
if c5_liver is not None: print(f"  C5 Liver: {c5_liver.shape}")
if c5_blood is not None: print(f"  C5 Blood: {c5_blood.shape}")

# --- C7: Correlations ---
c7_corr = None
for f in (os.listdir(C7_DIR) if os.path.exists(C7_DIR) else []):
    if 'corr' in f.lower() and f.endswith('.csv'):
        c7_corr = pd.read_csv(f'{C7_DIR}/{f}')
        print(f"  C7 correlations: {c7_corr.shape} from {f}")
        break

# --- C3 gene list ---
c3_genelist_path = f'{C3_DIR}/C3_gene_list_196genes.csv'
c3_genelist = pd.read_csv(c3_genelist_path) if os.path.exists(c3_genelist_path) else None
if c3_genelist is not None: print(f"  C3 gene list: {c3_genelist.shape}")

# --- C8 ---
c8_it = pd.read_csv(f'{C8_DIR}/C8_IT_specific_genes.csv') if os.path.exists(f'{C8_DIR}/C8_IT_specific_genes.csv') else None
c8_cr = pd.read_csv(f'{C8_DIR}/C8_CR_scar_genes.csv') if os.path.exists(f'{C8_DIR}/C8_CR_scar_genes.csv') else None
if c8_it is not None: print(f"  C8 IT-specific: {c8_it.shape}")
if c8_cr is not None: print(f"  C8 CR scar: {c8_cr.shape}")

print("✅ Data loading complete")

  LOADING EXISTING C3-C8 RESULTS
  C3 stats: (16464, 16) — columns: ['tissue', 'lineage', 'gene', 'comparison', 'grp1', 'grp2', 'n1', 'n2', 'mean_grp1', 'mean_grp2']
  C4 Liver: (1274, 15)
  C4 Blood: (1274, 15)
  C5 Liver: (7252, 17)
  C5 Blood: (7252, 17)
  C7 correlations: (1470, 8) from C7_gene_gene_correlations.csv
  C3 gene list: (196, 4)
  C8 IT-specific: (122, 7)
  C8 CR scar: (114, 7)
✅ Data loading complete


In [8]:
# ============================================================
# CELL 1: DATA LOADING
# ============================================================
print("\n" + "=" * 70)
print("  CELL 1: DATA LOADING")
print("=" * 70)

# --- Auto-detect column names (GSE182159 실제 컬럼에 완벽 대응) ---
print("  Available columns:", list(adata.obs.columns))

# Tissue column
tissue_col = None
for c in ['tissue', 'Tissue', 'tissue_type']:
    if c in adata.obs.columns:
        tissue_col = c
        break
assert tissue_col is not None

# Stage column
stage_col = None
for c in ['Stage', 'stage', 'disease_stage', 'disease_group', 'group', 'condition']:
    if c in adata.obs.columns:
        stage_col = c
        break
assert stage_col is not None

# Donor column ← 여기서 'sample'이 실제 donor ID임을 명확히 우선
donor_col = None
for c in ['sample', 'donor', 'sample_id', 'donor_id', 'orig.ident', 'patient', 'subject', 'Patient', 'Donor']:
    if c in adata.obs.columns:
        donor_col = c
        break
assert donor_col is not None, f"❌ No donor column found! Available: {list(adata.obs.columns)}"

# Lineage column ← 'major_lineage' 추가 (오류 원인 해결)
lineage_col = None
for c in ['major_lineage', 'lineage', 'Lineage', 'cell_lineage', 'celltype_major', 'gut2021_subcluster_v2']:
    if c in adata.obs.columns:
        lineage_col = c
        break
assert lineage_col is not None, f"❌ No lineage column found! Available: {list(adata.obs.columns)}"

print(f"✅ 자동 탐지 성공 → tissue={tissue_col}, stage={stage_col}, donor={donor_col}, lineage={lineage_col}")
print(f"  Tissues: {adata.obs[tissue_col].value_counts().to_dict()}")
print(f"  Stages: {adata.obs[stage_col].value_counts().to_dict()}")
print(f"  Lineages: {sorted(adata.obs[lineage_col].unique())}")
print(f"  Donors: {adata.obs[donor_col].nunique()} unique")



  CELL 1: DATA LOADING
  Available columns: ['sample', 'tissue', 'Stage', 'IT_cluster_21', 'IT_cluster_23', 'IT_cluster_25', 'IT_nk_collapse', 'IT_IT_signature', 'GSM_ID', 'IT_score_v2', 'IT_score_v3', 'IT_score_v4', 'IT_signature_final', 'IT_like', 'PW_mTOR_signaling', 'PW_glycolysis', 'PW_oxidative_phosphorylation', 'PW_nk_cell_cytotoxicity', 'PW_il15_signaling', 'PW_b_cell_differentiation', 'leiden', 'gut2021_subcluster', 'major_lineage', 'gut2021_subcluster_v2', 'TCR_clone.id', 'TCR_v_gene.x', 'TCR_j_gene.x', 'TCR_cdr3_nt.x', 'TCR_CType', 'BCR_clone.id', 'BCR_v_gene', 'BCR_j_gene', 'BCR_cdr3_nt', 'BCR_CType', 'lineage_std']
✅ 자동 탐지 성공 → tissue=tissue, stage=Stage, donor=sample, lineage=major_lineage
  Tissues: {'Blood': 136408, 'Liver': 106592}
  Stages: {'IA': 62545, 'IT': 49179, 'AR': 45452, 'CR': 43245, 'NL': 42579}
  Lineages: ['B', 'CD4_T', 'CD8_T', 'Myeloid', 'NK', 'PlasmaB', 'gdT']
  Donors: 46 unique


## Cell 3. Fix 1 — FDR Correction (Benjamini-Hochberg)

**Most critical fix.** Applies FDR correction to all statistical tests:
- C3: ~16,464 tests | C4: ~2,184 tests | C5: ~14,504 tests | C7: 1,470 correlations

FDR is applied **per tissue** (Liver and Blood are independent analyses).

In [9]:
# ============================================================
#  FIX 1: FDR (BENJAMINI-HOCHBERG) CORRECTION
# ============================================================
print("=" * 70)
print("  FIX 1: FDR (BENJAMINI-HOCHBERG) CORRECTION")
print("=" * 70)

def apply_fdr_correction(df, p_col='p_value', method='fdr_bh'):
    """Apply BH FDR correction. Returns df with q_value and fdr_significant."""
    df = df.copy()
    valid_mask = df[p_col].notna() & (df[p_col] > 0)
    p_vals = df.loc[valid_mask, p_col].values
    if len(p_vals) == 0:
        df['q_value'] = np.nan
        df['fdr_significant'] = False
        return df
    rejected, q_vals, _, _ = multipletests(p_vals, alpha=0.05, method=method)
    df.loc[valid_mask, 'q_value'] = q_vals
    df.loc[valid_mask, 'fdr_significant'] = rejected
    df['q_value'] = df['q_value'].fillna(1.0)
    df['fdr_significant'] = df['fdr_significant'].fillna(False)
    return df

def fdr_summary(df, label, p_col='p_value'):
    """Print FDR correction summary."""
    total = len(df[df[p_col].notna()])
    nominal = (df[p_col] < 0.05).sum()
    fdr_sig = df['fdr_significant'].sum() if 'fdr_significant' in df.columns else 0
    survival = fdr_sig / nominal * 100 if nominal > 0 else 0
    fdr_q10 = (df['q_value'] < 0.10).sum() if 'q_value' in df.columns else 0
    print(f"  {label}:")
    print(f"    Total tests: {total} | Nominal p<0.05: {nominal} ({nominal/total*100:.1f}%)")
    print(f"    FDR q<0.05: {fdr_sig} ({fdr_sig/total*100:.1f}%) | FDR q<0.10: {fdr_q10}")
    print(f"    Survival rate: {survival:.1f}%")
    return {'total': total, 'nominal': nominal, 'fdr_005': fdr_sig, 'fdr_010': fdr_q10}

def detect_p_col(df):
    """Auto-detect p-value column name."""
    for c in ['p_value', 'pvalue', 'p-value', 'p_val', 'P_value', 'pval']:
        if c in df.columns: return c
    for c in df.columns:
        if 'p' in c.lower() and 'val' in c.lower(): return c
    return None

fdr_results = {}

# --- 1A: C3 FDR ---
print("\n── FDR 1A: C3 Statistics (196 genes) ──")
if c3_stats is not None:
    p_col = detect_p_col(c3_stats)
    if p_col:
        print(f"  p-value column: \'{p_col}\'")
        for tissue in ['Liver', 'Blood']:
            t_cols = [c for c in c3_stats.columns if 'tissue' in c.lower()]
            if t_cols:
                subset = c3_stats[c3_stats[t_cols[0]] == tissue].copy()
            else:
                n = len(c3_stats)
                subset = c3_stats.iloc[:n//2].copy() if tissue == 'Liver' else c3_stats.iloc[n//2:].copy()
            subset = apply_fdr_correction(subset, p_col=p_col)
            fdr_results[f'C3_{tissue}'] = fdr_summary(subset, f'C3 {tissue}', p_col=p_col)
            subset.to_csv(f'{C9_DIR}/Fix1_FDR/C3_{tissue.lower()}_FDR.csv', index=False)
    else:
        print(f"  ⚠️ Cannot find p-value column. Columns: {list(c3_stats.columns)}")
else:
    print("  ⚠️ C3 stats not loaded")

# --- 1B: C4 FDR ---
print("\n── FDR 1B: C4 Pathway Statistics ──")
for tissue, df in [('Liver', c4_liver), ('Blood', c4_blood)]:
    if df is not None:
        p_col = detect_p_col(df)
        if p_col:
            df_fdr = apply_fdr_correction(df, p_col=p_col)
            fdr_results[f'C4_{tissue}'] = fdr_summary(df_fdr, f'C4 {tissue}', p_col=p_col)
            df_fdr.to_csv(f'{C9_DIR}/Fix1_FDR/C4_{tissue.lower()}_FDR.csv', index=False)
        else:
            print(f"  ⚠️ C4 {tissue}: p-value column not found. Cols: {list(df.columns)}")
    else:
        print(f"  ⚠️ C4 {tissue}: Not loaded")

# --- 1C: C5 FDR ---
print("\n── FDR 1C: C5 Gene Statistics (148 genes) ──")
for tissue, df in [('Liver', c5_liver), ('Blood', c5_blood)]:
    if df is not None:
        p_col = detect_p_col(df)
        if p_col:
            df_fdr = apply_fdr_correction(df, p_col=p_col)
            fdr_results[f'C5_{tissue}'] = fdr_summary(df_fdr, f'C5 {tissue}', p_col=p_col)
            df_fdr.to_csv(f'{C9_DIR}/Fix1_FDR/C5_{tissue.lower()}_FDR.csv', index=False)
        else:
            print(f"  ⚠️ C5 {tissue}: p-value column not found")
    else:
        print(f"  ⚠️ C5 {tissue}: Not loaded")

# --- 1D: C7 Correlation FDR ---
print("\n── FDR 1D: C7 Correlations ──")
if c7_corr is not None:
    p_col = detect_p_col(c7_corr)
    if p_col:
        c7_fdr = apply_fdr_correction(c7_corr, p_col=p_col)
        fdr_results['C7'] = fdr_summary(c7_fdr, 'C7 Correlations', p_col=p_col)
        c7_fdr.to_csv(f'{C9_DIR}/Fix1_FDR/C7_correlations_FDR.csv', index=False)
    else:
        print(f"  ⚠️ C7: p-value column not found")
else:
    print("  ⚠️ C7 correlations not loaded")

# --- Summary ---
print("\n" + "─" * 50)
print("  FDR CORRECTION SUMMARY")
print("─" * 50)
summary_rows = []
for key, val in fdr_results.items():
    surv = val['fdr_005'] / val['nominal'] * 100 if val['nominal'] > 0 else 0
    summary_rows.append({'Analysis': key, 'Total': val['total'], 'Nominal': val['nominal'],
                         'FDR_q005': val['fdr_005'], 'FDR_q010': val['fdr_010'], 'Survival': surv})
    print(f"  {key}: {val['nominal']} nominal → {val['fdr_005']} FDR(q<0.05) [{surv:.0f}%]")

if summary_rows:
    pd.DataFrame(summary_rows).to_csv(f'{C9_DIR}/Fix1_FDR/FDR_summary_report.csv', index=False)
print("\n✅ Fix 1 complete")

  FIX 1: FDR (BENJAMINI-HOCHBERG) CORRECTION

── FDR 1A: C3 Statistics (196 genes) ──
  p-value column: 'p_value'
  C3 Liver:
    Total tests: 8232 | Nominal p<0.05: 583 (7.1%)
    FDR q<0.05: 0 (0.0%) | FDR q<0.10: 0
    Survival rate: 0.0%
  C3 Blood:
    Total tests: 8232 | Nominal p<0.05: 859 (10.4%)
    FDR q<0.05: 0 (0.0%) | FDR q<0.10: 0
    Survival rate: 0.0%

── FDR 1B: C4 Pathway Statistics ──
  C4 Liver:
    Total tests: 1274 | Nominal p<0.05: 108 (8.5%)
    FDR q<0.05: 0 (0.0%) | FDR q<0.10: 0
    Survival rate: 0.0%
  C4 Blood:
    Total tests: 1274 | Nominal p<0.05: 130 (10.2%)
    FDR q<0.05: 0 (0.0%) | FDR q<0.10: 0
    Survival rate: 0.0%

── FDR 1C: C5 Gene Statistics (148 genes) ──
  C5 Liver:
    Total tests: 7252 | Nominal p<0.05: 398 (5.5%)
    FDR q<0.05: 0 (0.0%) | FDR q<0.10: 0
    Survival rate: 0.0%
  C5 Blood:
    Total tests: 7252 | Nominal p<0.05: 610 (8.4%)
    FDR q<0.05: 0 (0.0%) | FDR q<0.10: 0
    Survival rate: 0.0%

── FDR 1D: C7 Correlations ──
  

## Cell 4. Fix 2 — Add 3 Missing Pathways (Part A: Load h5ad)

Missing pathways: `antigen_presentation`, `type1_ifn`, `tgfb_signaling`

> ⚠️ If column auto-detection fails, **manually set** `tissue_col`, `lineage_col`, `donor_col`, `stage_col`.

In [10]:
# ============================================================
#  FIX 2A: LOAD h5ad & DETECT COLUMNS
# ============================================================
print("=" * 70)
print("  FIX 2A: LOAD DATA & DEFINE NEW PATHWAYS")
print("=" * 70)

import scanpy as sc
import scipy.sparse as sp

print("  Loading h5ad...")
adata = sc.read_h5ad(DATA_PATH)
print(f"  adata: {adata.shape[0]} cells × {adata.shape[1]} genes")
print(f"  obs columns: {list(adata.obs.columns)}")

# --- 3 new gene sets ---
new_gene_sets = {
    'antigen_presentation': ['HLA-DRA','HLA-DRB1','HLA-DPB1','HLA-DPA1','HLA-DQB1',
                             'CD74','B2M','TAP1','TAP2','CIITA'],
    'type1_ifn': ['MX1','ISG15','STAT1','STAT2','IRF3','IRF7',
                  'IFNAR1','OAS1','IFI44L','IFIT1'],
    'tgfb_signaling': ['TGFB1','TGFB2','TGFBR1','TGFBR2',
                       'SMAD2','SMAD3','SMAD4','SMAD7','TGFBI','LTBP1']
}

available_genes = set(adata.var_names)
print("\n  Gene availability:")
for pw, genes in new_gene_sets.items():
    found = [g for g in genes if g in available_genes]
    missing = [g for g in genes if g not in available_genes]
    print(f"  {pw}: {len(found)}/{len(genes)}", end="")
    if missing: print(f"  Missing: {missing}", end="")
    print()
    new_gene_sets[pw] = found

# --- Detect columns ---
tissue_col = lineage_col = donor_col = stage_col = None

for col in adata.obs.columns:
    cl = col.lower()
    if 'tissue' in cl or 'source' in cl or 'sample_type' in cl: tissue_col = col
    if 'lineage' in cl or 'cell_type' in cl or 'major' in cl: lineage_col = col
    if ('donor' in cl or 'patient' in cl) and donor_col is None: donor_col = col
    if 'stage' in cl or 'group' in cl or 'condition' in cl or 'disease' in cl: stage_col = col

# Fallback: inspect values
if tissue_col is None:
    for c in adata.obs.columns:
        if {'Liver','Blood'}.issubset(set(adata.obs[c].astype(str).unique())):
            tissue_col = c; break
if lineage_col is None:
    for c in adata.obs.columns:
        if any('Myeloid' in str(v) or 'CD4' in str(v) for v in adata.obs[c].unique()):
            lineage_col = c; break
if stage_col is None:
    for c in adata.obs.columns:
        if {'NL','IT','IA','AR','CR'}.issubset(set(adata.obs[c].astype(str).unique())):
            stage_col = c; break

print(f"\n  Detected columns:")
print(f"    tissue_col  = '{tissue_col}'")
print(f"    lineage_col = '{lineage_col}'")
print(f"    donor_col   = '{donor_col}'")
print(f"    stage_col   = '{stage_col}'")

# ┌──────────────────────────────────────────────────────────┐
# │  ⚠️ IF ANY IS None, MANUALLY SET HERE:                  │
# │  tissue_col  = 'your_column'                            │
# │  lineage_col = 'your_column'                            │
# │  donor_col   = 'your_column'                            │
# │  stage_col   = 'your_column'                            │
# └──────────────────────────────────────────────────────────┘

lineage_map = {'Myeloid':'Myeloid','CD4_T':'CD4_T','CD4T':'CD4_T',
               'CD8_T':'CD8_T','CD8T':'CD8_T','NK':'NK','B':'B',
               'PlasmaB':'PlasmaB','gdT':'gdT'}
adata.obs['lineage_std'] = adata.obs[lineage_col].map(lambda x: lineage_map.get(x, x))

LINEAGES = ['Myeloid','CD4_T','CD8_T','NK','B','PlasmaB']
TISSUES = ['Liver','Blood']
COMPARISONS = [('NL','IT'),('NL','IA'),('NL','AR'),('NL','CR'),
               ('IA','AR'),('IT','IA'),('IT','CR')]
print("✅ Ready")

  FIX 2A: LOAD DATA & DEFINE NEW PATHWAYS
  Loading h5ad...
  adata: 243000 cells × 24452 genes
  obs columns: ['sample', 'tissue', 'Stage', 'IT_cluster_21', 'IT_cluster_23', 'IT_cluster_25', 'IT_nk_collapse', 'IT_IT_signature', 'GSM_ID', 'IT_score_v2', 'IT_score_v3', 'IT_score_v4', 'IT_signature_final', 'IT_like', 'PW_mTOR_signaling', 'PW_glycolysis', 'PW_oxidative_phosphorylation', 'PW_nk_cell_cytotoxicity', 'PW_il15_signaling', 'PW_b_cell_differentiation', 'leiden', 'gut2021_subcluster', 'major_lineage', 'gut2021_subcluster_v2', 'TCR_clone.id', 'TCR_v_gene.x', 'TCR_j_gene.x', 'TCR_cdr3_nt.x', 'TCR_CType', 'BCR_clone.id', 'BCR_v_gene', 'BCR_j_gene', 'BCR_cdr3_nt', 'BCR_CType']

  Gene availability:
  antigen_presentation: 10/10
  type1_ifn: 10/10
  tgfb_signaling: 10/10

  Detected columns:
    tissue_col  = 'tissue'
    lineage_col = 'major_lineage'
    donor_col   = 'None'
    stage_col   = 'Stage'
✅ Ready


## Cell 5. Fix 2 — Score 3 New Pathways

In [12]:
# ============================================================
#  FIX 2B: SCORE & STAT TEST
# ============================================================
print("=" * 70)
print("  FIX 2B: SCORING 3 NEW PATHWAYS")
print("=" * 70)

# --- PATCH: Fix missing donor column ---
if 'donor_col' not in globals() or donor_col is None:
    donor_col = 'sample'
    print(f"  ⚠️ Automatically fixed donor_col='{donor_col}'")

def score_pathway_donor(adata, genes, tissue, lineage, stage):
    """Score pathway as mean expression per donor."""
    # Ensure we use the global donor_col
    global donor_col

    mask = ((adata.obs[tissue_col]==tissue) & (adata.obs['lineage_std']==lineage) & (adata.obs[stage_col]==stage))
    subset = adata[mask]
    if subset.shape[0]==0: return {}
    gene_idx = [i for i,g in enumerate(adata.var_names) if g in genes]
    if not gene_idx: return {}
    X = subset.X[:, gene_idx]
    if sp.issparse(X): X = X.toarray()
    cell_scores = np.mean(X, axis=1)
    donors = subset.obs[donor_col].values
    result = {}
    for d in np.unique(donors):
        dm = donors==d
        if dm.sum()>=5: result[d] = np.mean(cell_scores[dm])
    return result

all_results = []
total = len(new_gene_sets)*len(LINEAGES)*len(COMPARISONS)*len(TISSUES)
done = 0

for pw_name, pw_genes in new_gene_sets.items():
    print(f"\n── {pw_name} ({len(pw_genes)} genes) ──")
    for tissue in TISSUES:
        for lineage in LINEAGES:
            for grp1, grp2 in COMPARISONS:
                s1 = score_pathway_donor(adata, pw_genes, tissue, lineage, grp1)
                s2 = score_pathway_donor(adata, pw_genes, tissue, lineage, grp2)
                v1, v2 = list(s1.values()), list(s2.values())

                if len(v1)>=2 and len(v2)>=2:
                    _, pval = stats.mannwhitneyu(v1, v2, alternative='two-sided')
                    m1, m2 = np.mean(v1), np.mean(v2)
                    pct = ((m2-m1)/m1*100) if m1!=0 else np.nan
                    d = '↑' if m2>m1 else '↓'
                    nc = sum(1 for a in v1 for b in v2 if (m2>m1 and b>a) or (m2<m1 and b<a))
                    nt = len(v1)*len(v2)
                    sig = '★' if pval<0.05 else ('†' if pval<0.10 else '')
                    all_results.append(dict(pathway=pw_name, tissue=tissue, lineage=lineage,
                        comparison=f'{grp1}→{grp2}', n_grp1=len(v1), n_grp2=len(v2),
                        mean_grp1=round(m1,6), mean_grp2=round(m2,6), pct_change=round(pct,1),
                        direction=d, p_value=round(pval,6), significance=sig,
                        consistency=f'{nc}/{nt}', n_genes=len(pw_genes)))
                else:
                    all_results.append(dict(pathway=pw_name, tissue=tissue, lineage=lineage,
                        comparison=f'{grp1}→{grp2}', n_grp1=len(v1), n_grp2=len(v2),
                        mean_grp1=np.nan, mean_grp2=np.nan, pct_change=np.nan,
                        direction='', p_value=np.nan, significance='',
                        consistency='', n_genes=len(pw_genes)))
                done += 1
                if done % 50 == 0: print(f"  {done}/{total} ({done/total*100:.0f}%)")

df_new_pw = pd.DataFrame(all_results)
df_new_pw_fdr = apply_fdr_correction(df_new_pw, p_col='p_value')
df_new_pw_fdr.to_csv(f'{C9_DIR}/Fix2_NewPathways/C4_3new_pathways_all_results.csv', index=False)

# Key results
print("\n" + "─" * 60)
print("  NL→IT RESULTS:")
print("─" * 60)
nl_it = df_new_pw_fdr[df_new_pw_fdr['comparison']=='NL→IT']
for _, r in nl_it[nl_it['p_value']<0.05].sort_values('p_value').iterrows():
    fdr = '(FDR★)' if r.get('fdr_significant', False) else '(nominal)'
    print(f"  {r['significance']} {r['pathway']:25s} | {r['tissue']:5s} {r['lineage']:8s} | "
          f"{r['direction']}{abs(r['pct_change']):.1f}% p={r['p_value']:.4f} {fdr}")

print("\n✅ Fix 2 complete")

  FIX 2B: SCORING 3 NEW PATHWAYS
  ⚠️ Automatically fixed donor_col='sample'

── antigen_presentation (10 genes) ──
  50/252 (20%)

── type1_ifn (10 genes) ──
  100/252 (40%)
  150/252 (60%)

── tgfb_signaling (10 genes) ──
  200/252 (79%)
  250/252 (99%)

────────────────────────────────────────────────────────────
  NL→IT RESULTS:
────────────────────────────────────────────────────────────
  ★ antigen_presentation      | Liver Myeloid  | ↑47.9% p=0.0022 (nominal)
  ★ antigen_presentation      | Blood Myeloid  | ↑50.0% p=0.0061 (nominal)
  ★ type1_ifn                 | Blood PlasmaB  | ↑119.9% p=0.0061 (nominal)
  ★ type1_ifn                 | Blood Myeloid  | ↑173.2% p=0.0061 (nominal)
  ★ tgfb_signaling            | Blood PlasmaB  | ↑214.7% p=0.0061 (nominal)
  ★ tgfb_signaling            | Blood Myeloid  | ↑89.3% p=0.0061 (nominal)
  ★ antigen_presentation      | Liver PlasmaB  | ↑87.4% p=0.0173 (nominal)
  ★ antigen_presentation      | Blood PlasmaB  | ↑35.4% p=0.0242 (nominal)
 

## Cell 6. Fix 3 — "99999%" Extreme Fold-Change Review

- 🟢 All target-group donors expressing → reliable
- 🟡 Zero baseline, ≥2 donors expressing → caution
- 🔴 Zero baseline, ≤1 donor expressing → unreliable (single donor artifact)

In [13]:
# ============================================================
#  FIX 3: EXTREME FOLD-CHANGE DONOR-LEVEL REVIEW
# ============================================================
print("=" * 70)
print("  FIX 3: EXTREME FOLD-CHANGE REVIEW")
print("=" * 70)

extreme_genes = {
    ('MEFV','CD8_T','Liver'), ('MEFV','B','Blood'), ('MEFV','NK','Blood'),
    ('NLRP3','B','Blood'), ('NLRC4','CD8_T','Blood'),
    ('COL1A1','CD8_T','Blood'), ('FN1','CD4_T','Blood'),
    ('AICDA','B','Liver'), ('AICDA','B','Blood'),
    ('PRF1','Myeloid','Liver'), ('TOX','Myeloid','Liver'),
}

print(f"  Reviewing {len(extreme_genes)} gene-lineage-tissue combos\n")
extreme_review = []

for gene, lineage, tissue in sorted(extreme_genes):
    if gene not in adata.var_names:
        print(f"  ⚠️ {gene} not in dataset"); continue
    gene_idx = list(adata.var_names).index(gene)
    for stage in ['NL','IT','IA','AR','CR']:
        mask = ((adata.obs[tissue_col]==tissue) & (adata.obs['lineage_std']==lineage) & (adata.obs[stage_col]==stage))
        subset = adata[mask]
        if subset.shape[0]==0: continue
        X = subset.X[:, gene_idx]
        if sp.issparse(X): X = X.toarray().flatten()
        else: X = np.array(X).flatten()
        donors = subset.obs[donor_col].values
        for d in np.unique(donors):
            dm = donors==d
            d_expr = X[dm]
            nc = len(d_expr); ne = (d_expr>0).sum()
            extreme_review.append(dict(gene=gene, lineage=lineage, tissue=tissue,
                stage=stage, donor=d, n_cells=nc, n_expressing=ne,
                pct_expressing=round(ne/nc*100,1) if nc>0 else 0,
                mean_expression=round(np.mean(d_expr),6),
                mean_if_expressed=round(np.mean(d_expr[d_expr>0]),6) if ne>0 else 0))

df_extreme = pd.DataFrame(extreme_review)
df_extreme.to_csv(f'{C9_DIR}/Fix3_ExtremeFC/extreme_foldchange_donor_review.csv', index=False)

# Summary
for (gene, lineage, tissue), grp in df_extreme.groupby(['gene','lineage','tissue']):
    print(f"\n  {gene:8s} {lineage:8s} {tissue:6s}:")
    for stage in ['NL','IT','IA','AR','CR']:
        sd = grp[grp['stage']==stage]
        if len(sd)==0: continue
        nd = len(sd); ne = (sd['n_expressing']>0).sum()
        print(f"    {stage}: {nd} donors, {ne}/{nd} expressing, "
              f"avg {sd['pct_expressing'].mean():.1f}% cells+, mean={sd['mean_expression'].mean():.4f}")

# Reliability flags
print("\n  ⚠️ RELIABILITY FLAGS:")
for (gene, lineage, tissue), grp in df_extreme.groupby(['gene','lineage','tissue']):
    cr = grp[grp['stage']=='CR']; nl = grp[grp['stage']=='NL']
    if len(cr)>0 and len(nl)>0:
        cre = (cr['n_expressing']>0).sum(); nle = (nl['n_expressing']>0).sum()
        if nle==0 and cre<=1: print(f"  🔴 {gene}/{lineage}/{tissue}: NL 0/{len(nl)}, CR {cre}/{len(cr)} — UNRELIABLE")
        elif nle==0 and cre>=2: print(f"  🟡 {gene}/{lineage}/{tissue}: NL 0/{len(nl)}, CR {cre}/{len(cr)} — CAUTION")
        elif cre==len(cr): print(f"  🟢 {gene}/{lineage}/{tissue}: All {cre} CR donors expressing — RELIABLE")
print("\n✅ Fix 3 complete")

  FIX 3: EXTREME FOLD-CHANGE REVIEW
  Reviewing 11 gene-lineage-tissue combos


  AICDA    B        Blood :
    NL: 5 donors, 1/5 expressing, avg 0.0% cells+, mean=0.0003
    IT: 7 donors, 6/7 expressing, avg 0.8% cells+, mean=0.0124
    IA: 4 donors, 4/4 expressing, avg 1.0% cells+, mean=0.0162
    AR: 4 donors, 4/4 expressing, avg 1.1% cells+, mean=0.0175
    CR: 3 donors, 1/3 expressing, avg 0.2% cells+, mean=0.0025

  AICDA    B        Liver :
    NL: 6 donors, 0/6 expressing, avg 0.0% cells+, mean=0.0000
    IT: 5 donors, 3/5 expressing, avg 0.6% cells+, mean=0.0079
    IA: 5 donors, 1/5 expressing, avg 0.1% cells+, mean=0.0009
    AR: 3 donors, 2/3 expressing, avg 1.3% cells+, mean=0.0169
    CR: 3 donors, 0/3 expressing, avg 0.0% cells+, mean=0.0000

  COL1A1   CD8_T    Blood :
    NL: 5 donors, 0/5 expressing, avg 0.0% cells+, mean=0.0000
    IT: 7 donors, 0/7 expressing, avg 0.0% cells+, mean=0.0000
    IA: 4 donors, 1/4 expressing, avg 0.1% cells+, mean=0.0026
    AR: 4 donor

## Cell 7. Fix 4 — C3 (196) vs C5 (148) Gene Reconciliation

In [14]:
# ============================================================
#  FIX 4: C3 vs C5 GENE RECONCILIATION
# ============================================================
print("=" * 70)
print("  FIX 4: GENE RECONCILIATION")
print("=" * 70)

c3_genes, c5_genes = set(), set()

if c3_genelist is not None:
    c3_genes = set(c3_genelist.iloc[:,0].dropna().unique())
    print(f"  C3: {len(c3_genes)} genes")
elif c3_stats is not None:
    gc = [c for c in c3_stats.columns if 'gene' in c.lower()]
    if gc: c3_genes = set(c3_stats[gc[0]].dropna().unique()); print(f"  C3 from stats: {len(c3_genes)}")

if c5_liver is not None:
    gc = [c for c in c5_liver.columns if 'gene' in c.lower()]
    if gc: c5_genes = set(c5_liver[gc[0]].dropna().unique()); print(f"  C5: {len(c5_genes)} genes")

if len(c3_genes)>0 and len(c5_genes)>0:
    c3_only = c3_genes - c5_genes
    shared = c3_genes & c5_genes
    print(f"\n  Shared: {len(shared)} | C3-only: {len(c3_only)} | C5-only: {len(c5_genes-c3_genes)}")

    if c3_only:
        print(f"\n  C3-only genes ({len(c3_only)}): {sorted(c3_only)}")

    key_genes = ['AICDA','SOCS1','SOCS3','JCHAIN','IL1RN','RICTOR',
                 'MX1','ISG15','STAT2','IRF3','SERPINE1','ID3']
    print(f"\n  {'Gene':12s} {'C3':4s} {'C5':4s} Status")
    print("  " + "─" * 40)
    rows = []
    for g in key_genes:
        ic3, ic5 = g in c3_genes, g in c5_genes
        st = "✅ Both" if ic3 and ic5 else ("⚠️ C3-only" if ic3 else ("⚠️ C5-only" if ic5 else "🔴 Neither"))
        print(f"  {g:12s} {'Y' if ic3 else 'N':4s} {'Y' if ic5 else 'N':4s} {st}")
        rows.append(dict(gene=g, in_C3=ic3, in_C5=ic5, status=st))

    # Save
    all_g = sorted(c3_genes | c5_genes)
    rec = pd.DataFrame({'gene': all_g, 'in_C3': [g in c3_genes for g in all_g],
                         'in_C5': [g in c5_genes for g in all_g]})
    rec['source'] = rec.apply(lambda r: 'Both' if r['in_C3'] and r['in_C5'] else ('C3-only' if r['in_C3'] else 'C5-only'), axis=1)
    rec.to_csv(f'{C9_DIR}/Fix4_GeneReconcile/gene_reconciliation_C3_vs_C5.csv', index=False)
    pd.DataFrame(rows).to_csv(f'{C9_DIR}/Fix4_GeneReconcile/key_claims_gene_source.csv', index=False)
else:
    print("  ⚠️ Could not load both gene lists")
print("\n✅ Fix 4 complete")

  FIX 4: GENE RECONCILIATION
  C3: 196 genes
  C5: 148 genes

  Shared: 81 | C3-only: 115 | C5-only: 67

  C3-only genes (115): ['ACADVL', 'AICDA', 'AKT1', 'APC', 'ATR', 'AXIN1', 'B2M', 'BACH2', 'BAX', 'BBC3', 'BCL2L1', 'BID', 'BIRC3', 'BRCA1', 'CASP4', 'CASP5', 'CCL3', 'CCL4', 'CCL5', 'CCR2', 'CCR5', 'CD19', 'CD74', 'CD79A', 'CD79B', 'CFLAR', 'CGAS', 'CHEK1', 'CHEK2', 'CIITA', 'CISH', 'COX5A', 'CTNNB1', 'CX3CR1', 'CXCL10', 'CXCR3', 'CXCR4', 'CXCR5', 'DDX58', 'FGFBP2', 'GSDMD', 'GZMH', 'GZMM', 'HADHA', 'HGF', 'HLA-A', 'HLA-B', 'HLA-C', 'HLA-DPA1', 'HLA-DPB1', 'HLA-DRA', 'HLA-DRB1', 'IFIH1', 'IFIT1', 'IFNAR1', 'IFNAR2', 'IFNB1', 'IL10', 'IL1B', 'IL1RN', 'IRF3', 'IRF7', 'IRF9', 'ISG15', 'JAK2', 'JCHAIN', 'KIR2DL4', 'LDHA', 'LILRB1', 'LILRB2', 'LRRC32', 'MAVS', 'MCL1', 'MDM2', 'MET', 'MS4A1', 'MX1', 'MX2', 'MYD88', 'MZB1', 'NAIP', 'NDUFS1', 'NFATC1', 'NT5E', 'OAS1', 'PAX5', 'PFKFB3', 'PIAS1', 'PKM', 'SDC1', 'SIGLEC10', 'SLC2A1', 'SOCS1', 'SOCS3', 'STAT1', 'STAT2', 'STAT3', 'STAT4', 'STAT6

## Cell 8. Fix 5 — Bootstrap CI for C7 Correlations

1,000 bootstrap iterations + leave-one-out jackknife for top 16 correlations.

In [15]:
# ============================================================
#  FIX 5A: BOOTSTRAP CI
# ============================================================
print("=" * 70)
print("  FIX 5A: BOOTSTRAP CI")
print("=" * 70)

def get_donor_means(adata, gene, tissue, lineage):
    """Get donor-level mean expression."""
    mask = (adata.obs[tissue_col]==tissue) & (adata.obs['lineage_std']==lineage)
    subset = adata[mask]
    if gene not in adata.var_names or subset.shape[0]==0: return {}
    gi = list(adata.var_names).index(gene)
    X = subset.X[:, gi]
    if sp.issparse(X): X = X.toarray().flatten()
    else: X = np.array(X).flatten()
    donors = subset.obs[donor_col].values
    result = {}
    for d in np.unique(donors):
        dm = donors==d
        if dm.sum()>=5: result[d] = np.mean(X[dm])
    return result

TOP_CORR = [
    ('MT-CYB','MT-ND1','Blood','CD8_T',0.929), ('MT-CYB','MT-ND1','Blood','CD4_T',0.887),
    ('MT-CYB','MT-ND1','Blood','NK',0.879), ('JAK1','TGFBR2','Blood','CD4_T',0.883),
    ('JAK1','TGFBR2','Liver','CD4_T',0.871), ('JAK1','TGFBR2','Blood','NK',0.849),
    ('TFAM','DNMT1','Blood','Myeloid',0.857), ('TFAM','DNMT1','Blood','NK',0.832),
    ('TFAM','MTOR','Blood','B',0.832), ('JAK1','TGFBR2','Blood','B',0.830),
    ('MT-CYB','MT-ND1','Blood','B',0.821), ('TGFBR2','DNMT3A','Liver','NK',0.817),
    ('MT-ND2','MT-ND1','Blood','NK',0.848), ('MT-ND2','MT-ND1','Blood','CD8_T',0.823),
    ('MT-ND2','MT-ND1','Blood','CD4_T',0.814), ('JAK1','MTOR','Blood','Myeloid',0.812),
]

N_BOOT = 1000
np.random.seed(42)
boot_results = []

for i, (g1,g2,tis,lin,exp_rho) in enumerate(TOP_CORR):
    m1 = get_donor_means(adata, g1, tis, lin)
    m2 = get_donor_means(adata, g2, tis, lin)
    common = sorted(set(m1) & set(m2))
    if len(common)<5:
        print(f"  [{i+1:2d}] {g1}↔{g2} ({tis}/{lin}): {len(common)} donors — SKIP"); continue
    x = np.array([m1[d] for d in common])
    y = np.array([m2[d] for d in common])
    n = len(common)
    rho, pval = stats.spearmanr(x, y)

    # Bootstrap
    br = []
    for _ in range(N_BOOT):
        idx = np.random.choice(n, n, replace=True)
        try:
            r, _ = stats.spearmanr(x[idx], y[idx])
            if not np.isnan(r): br.append(r)
        except: pass
    br = np.array(br)
    ci_lo, ci_hi = np.percentile(br, 2.5), np.percentile(br, 97.5)

    # Jackknife
    jr = []
    for j in range(n):
        try:
            r, _ = stats.spearmanr(np.delete(x,j), np.delete(y,j))
            jr.append(r)
        except: jr.append(np.nan)
    jr = np.array(jr)
    infl = [common[j] for j in range(n) if j<len(jr) and abs(jr[j]-rho)>0.10]

    boot_results.append(dict(gene1=g1, gene2=g2, tissue=tis, lineage=lin,
        n_donors=n, rho=round(rho,4), rho_expected=exp_rho, p_value=round(pval,6),
        CI_lower=round(ci_lo,4), CI_upper=round(ci_hi,4), CI_width=round(ci_hi-ci_lo,4),
        jack_range=round(np.nanmax(jr)-np.nanmin(jr),4),
        n_influential=len(infl), influential=','.join(str(d) for d in infl) or 'None',
        robust=ci_lo>0.5))

    mk = '✅' if ci_lo>0.5 else ('🟡' if ci_lo>0.3 else '🔴')
    print(f"  [{i+1:2d}] {g1:8s}↔{g2:8s} {tis:5s}/{lin:8s}: "
          f"ρ={rho:.3f} [{ci_lo:.3f},{ci_hi:.3f}] p={pval:.4f} infl={len(infl)} {mk}")

df_bootstrap = pd.DataFrame(boot_results)
df_bootstrap.to_csv(f'{C9_DIR}/Fix5_BootstrapCI/C7_bootstrap_validation.csv', index=False)

nr = (df_bootstrap['robust']==True).sum()
print(f"\n  Robust (CI>0.5): {nr}/{len(df_bootstrap)}")
print(f"  Moderate: {((df_bootstrap['CI_lower']>0.3)&(df_bootstrap['CI_lower']<=0.5)).sum()}")
print(f"  Weak: {(df_bootstrap['CI_lower']<=0.3).sum()}")
print("\n✅ Fix 5A complete")

  FIX 5A: BOOTSTRAP CI
  [ 1] MT-CYB  ↔MT-ND1   Blood/CD8_T   : ρ=0.929 [0.816,0.975] p=0.0000 infl=0 ✅
  [ 2] MT-CYB  ↔MT-ND1   Blood/CD4_T   : ρ=0.887 [0.723,0.950] p=0.0000 infl=0 ✅
  [ 3] MT-CYB  ↔MT-ND1   Blood/NK      : ρ=0.879 [0.705,0.954] p=0.0000 infl=0 ✅
  [ 4] JAK1    ↔TGFBR2   Blood/CD4_T   : ρ=0.883 [0.707,0.958] p=0.0000 infl=0 ✅
  [ 5] JAK1    ↔TGFBR2   Liver/CD4_T   : ρ=0.871 [0.641,0.970] p=0.0000 infl=0 ✅
  [ 6] JAK1    ↔TGFBR2   Blood/NK      : ρ=0.849 [0.659,0.917] p=0.0000 infl=0 ✅
  [ 7] TFAM    ↔DNMT1    Blood/Myeloid : ρ=0.857 [0.593,0.968] p=0.0000 infl=0 ✅
  [ 8] TFAM    ↔DNMT1    Blood/NK      : ρ=0.832 [0.638,0.919] p=0.0000 infl=0 ✅
  [ 9] TFAM    ↔MTOR     Blood/B       : ρ=0.832 [0.625,0.916] p=0.0000 infl=0 ✅
  [10] JAK1    ↔TGFBR2   Blood/B       : ρ=0.830 [0.550,0.943] p=0.0000 infl=0 ✅
  [11] MT-CYB  ↔MT-ND1   Blood/B       : ρ=0.821 [0.588,0.917] p=0.0000 infl=0 ✅
  [12] TGFBR2  ↔DNMT3A   Liver/NK      : ρ=0.817 [0.622,0.917] p=0.0000 infl=0 ✅
  [13

## Cell 9. Fix 5 — Scatter Plots

In [16]:
# ============================================================
#  FIX 5B: SCATTER PLOTS
# ============================================================
import matplotlib; matplotlib.use('Agg')
import matplotlib.pyplot as plt
from matplotlib.lines import Line2D

n_plots = min(len(TOP_CORR), 16)
fig, axes = plt.subplots((n_plots+3)//4, 4, figsize=(20, 5*((n_plots+3)//4)))
axes = axes.flatten()

colors = {'NL':'#2ca02c','IT':'#d62728','IA':'#ff7f0e','AR':'#1f77b4','CR':'#9467bd'}
markers = {'NL':'o','IT':'s','IA':'D','AR':'^','CR':'v'}
smap = dict(zip(adata.obs[donor_col], adata.obs[stage_col]))

for i, (g1,g2,tis,lin,_) in enumerate(TOP_CORR[:n_plots]):
    ax = axes[i]
    m1 = get_donor_means(adata, g1, tis, lin)
    m2 = get_donor_means(adata, g2, tis, lin)
    common = sorted(set(m1) & set(m2))
    if len(common)<3:
        ax.set_title(f"{g1}↔{g2}\n{tis}/{lin}\nN/A", fontsize=9); ax.set_facecolor('white'); continue
    x = np.array([m1[d] for d in common])
    y = np.array([m2[d] for d in common])

    for di, d in enumerate(common):
        s = smap.get(d, '?')
        ax.scatter(x[di], y[di], c=colors.get(s,'gray'), marker=markers.get(s,'o'),
                  s=60, edgecolors='k', linewidth=0.5, zorder=5)
    z = np.polyfit(x, y, 1); ax.plot(np.linspace(x.min(),x.max(),50), np.poly1d(z)(np.linspace(x.min(),x.max(),50)),
                                     '--', color='gray', alpha=0.5)
    rho, pv = stats.spearmanr(x, y)
    mt = df_bootstrap[(df_bootstrap['gene1']==g1)&(df_bootstrap['gene2']==g2)&
                       (df_bootstrap['tissue']==tis)&(df_bootstrap['lineage']==lin)]
    ci = f"\n[{mt.iloc[0]['CI_lower']:.2f},{mt.iloc[0]['CI_upper']:.2f}]" if len(mt)>0 else ""
    ax.set_xlabel(g1, fontsize=9); ax.set_ylabel(g2, fontsize=9)
    ax.set_title(f"{g1}↔{g2}\n{tis}/{lin}\nρ={rho:.3f} p={pv:.4f}{ci}", fontsize=8, fontweight='bold')
    ax.set_facecolor('white')

for j in range(i+1, len(axes)): axes[j].set_visible(False)

leg = [Line2D([0],[0],marker='o',color='w',markerfacecolor=colors[s],markersize=8,
              label=s,markeredgecolor='k',markeredgewidth=0.5) for s in ['NL','IT','IA','AR','CR']]
fig.legend(handles=leg, loc='lower right', ncol=5, fontsize=10, bbox_to_anchor=(0.98,0.02))
plt.suptitle('C7 Correlations: Bootstrap Validation', fontsize=14, fontweight='bold', y=1.01)
plt.tight_layout()
plt.savefig(f'{C9_DIR}/Fix5_BootstrapCI/C7_scatter_plots.png', dpi=300, bbox_inches='tight', facecolor='white')
plt.show()
print("✅ Fix 5B complete")

✅ Fix 5B complete


## Cell 10. Summary Report

In [17]:
# ============================================================
#  SUMMARY REPORT
# ============================================================
print("=" * 70)
print("  C9 SUMMARY REPORT")
print("=" * 70)

print("\n[FIX 1] FDR CORRECTION")
for k, v in fdr_results.items():
    s = v['fdr_005']/v['nominal']*100 if v['nominal']>0 else 0
    e = '✅' if s>50 else ('🟡' if s>20 else '🔴')
    print(f"  {e} {k}: {v['nominal']}→{v['fdr_005']} (q<0.05) [{s:.0f}%]")

print("\n[FIX 2] 3 NEW PATHWAYS (NL→IT)")
for pw in new_gene_sets:
    d = df_new_pw_fdr[(df_new_pw_fdr['pathway']==pw)&(df_new_pw_fdr['comparison']=='NL→IT')]
    ns = (d['p_value']<0.05).sum()
    nf = d['fdr_significant'].sum() if 'fdr_significant' in d.columns else 0
    print(f"  {pw}: {ns} nominal / {nf} FDR")

print("\n[FIX 3] EXTREME FC → see Fix3_ExtremeFC/")
print("[FIX 4] GENE RECONCILE → see Fix4_GeneReconcile/")

print("\n[FIX 5] BOOTSTRAP CI")
if len(df_bootstrap)>0:
    print(f"  Robust: {(df_bootstrap['robust']==True).sum()}/{len(df_bootstrap)}")
    print(f"  Influential donors: {(df_bootstrap['n_influential']>0).sum()}")

print(f"""
  ALL OUTPUTS → {C9_DIR}
  ├── Fix1_FDR/           (FDR-corrected CSVs)
  ├── Fix2_NewPathways/   (3 new pathway scores)
  ├── Fix3_ExtremeFC/     (donor-level review)
  ├── Fix4_GeneReconcile/ (C3 vs C5 mapping)
  └── Fix5_BootstrapCI/   (bootstrap + scatter plots)

  ✅ C9 COMPLETE
  ▶ NEXT: Review FDR → Update Results → Generate Figures
""")

  C9 SUMMARY REPORT

[FIX 1] FDR CORRECTION
  🔴 C3_Liver: 583→0 (q<0.05) [0%]
  🔴 C3_Blood: 859→0 (q<0.05) [0%]
  🔴 C4_Liver: 108→0 (q<0.05) [0%]
  🔴 C4_Blood: 130→0 (q<0.05) [0%]
  🔴 C5_Liver: 398→0 (q<0.05) [0%]
  🔴 C5_Blood: 610→0 (q<0.05) [0%]
  ✅ C7: 517→298 (q<0.05) [58%]

[FIX 2] 3 NEW PATHWAYS (NL→IT)
  antigen_presentation: 4 nominal / 0 FDR
  type1_ifn: 4 nominal / 0 FDR
  tgfb_signaling: 3 nominal / 0 FDR

[FIX 3] EXTREME FC → see Fix3_ExtremeFC/
[FIX 4] GENE RECONCILE → see Fix4_GeneReconcile/

[FIX 5] BOOTSTRAP CI
  Robust: 15/16
  Influential donors: 0

  ALL OUTPUTS → /content/drive/MyDrive/ITLAS/results/version18-analysis/C9_method_fixes/
  ├── Fix1_FDR/           (FDR-corrected CSVs)
  ├── Fix2_NewPathways/   (3 new pathway scores)
  ├── Fix3_ExtremeFC/     (donor-level review)
  ├── Fix4_GeneReconcile/ (C3 vs C5 mapping)
  └── Fix5_BootstrapCI/   (bootstrap + scatter plots)

  ✅ C9 COMPLETE
  ▶ NEXT: Review FDR → Update Results → Generate Figures

